In [14]:
import pandas as pd


In [ ]:
roles = pd.read_csv('../data/raw/Montagna_Roles.csv')
roles.head()

,Node,Role,Relationship,Request
0,N0,cooperating witness,NaN,NaN
1,N1,NaN,conversation with 0,NaN
2,N2,boss family ''Barcellona Pozzo di Gotto'',NaN,NaN
3,N3,boss family ''Caltagirone'',NaN,NaN
4,N4,enterpreneur,NaN,NaN


In [16]:
role_text = roles['Role'].astype('string').str.strip()

# If the word family is present, whatever comes before it is the role.
# Otherwise, keep the original role text.
roles['family_role'] = role_text.str.extract(r'^(.*?)\s*family\b', expand=False)
roles['family_role'] = roles['family_role'].fillna(role_text)
roles['family_role'] = roles['family_role'].str.strip().replace('', pd.NA)

# If there are quotes, whatever is inside is the family name.
# Support both double quotes and doubled single quotes from the CSV.
family_from_quotes = role_text.str.extract(r'"([^"]+)"|\'\'([^\']+)\'\'', expand=True)
roles['family'] = family_from_quotes.bfill(axis=1).iloc[:, 0]
roles['family'] = roles['family'].fillna('Unknown')

# Preserve truly missing role values.
missing_role_mask = role_text.isna() | role_text.eq('')
roles.loc[missing_role_mask, ['family_role', 'family']] = pd.NA

roles[['Node', 'Role', 'family_role', 'family']].head(20)

,Node,Role,family_role,family
0,N0,cooperating witness,cooperating witness,Unknown
1,N1,NaN,<NA>,<NA>
2,N2,boss family ''Barcellona Pozzo di Gotto'',boss,Barcellona Pozzo di Gotto
3,N3,boss family ''Caltagirone'',boss,Caltagirone
4,N4,enterpreneur,enterpreneur,Unknown
5,N5,NaN,<NA>,<NA>
6,N6,NaN,<NA>,<NA>
7,N7,NaN,<NA>,<NA>
8,N8,NaN,<NA>,<NA>
9,N9,NaN,<NA>,<NA>


In [17]:
test_cases = {
    "boss family ''Caltagirone''": ('boss', 'Caltagirone'),
    'boss family "Caltagirone"': ('boss', 'Caltagirone'),
    "member of family ''San Mauro Castelverde''": ('member of', 'San Mauro Castelverde'),
    "Co-founder of family ''Batanesi''": ('Co-founder of', 'Batanesi'),
    'cooperating witness': ('cooperating witness', 'Unknown'),
    'executive family': ('executive', 'Unknown'),
}

for raw_role, expected in test_cases.items():
    sample = pd.Series([raw_role], dtype='string').str.strip()
    parsed_role = sample.str.extract(r'^(.*?)\s*family\b', expand=False).fillna(sample).iloc[0]
    family_match = sample.str.extract(r'"([^"]+)"|\'\'([^\']+)\'\'', expand=True)
    parsed_family = family_match.bfill(axis=1).iloc[0, 0]
    parsed_family = parsed_family if pd.notna(parsed_family) else 'Unknown'
    assert (parsed_role, parsed_family) == expected

roles.loc[[0, 2, 3, 11, 12, 17, 22, 25], ['Node', 'Role', 'family_role', 'family']]

,Node,Role,family_role,family
0,N0,cooperating witness,cooperating witness,Unknown
2,N2,boss family ''Barcellona Pozzo di Gotto'',boss,Barcellona Pozzo di Gotto
3,N3,boss family ''Caltagirone'',boss,Caltagirone
11,N11,boss Cosa Nostra in Messina,boss Cosa Nostra in Messina,Unknown
12,N12,member family ''Mistretta'',member,Mistretta
17,N18,executive family ''Mistretta'',executive,Mistretta
22,N23,executive family,executive,Unknown
25,N26,member of family ''San Mauro Castelverde'',member of,San Mauro Castelverde


In [18]:
roles[['family_role', 'family']].value_counts(dropna=False).head(20)

family_role                  family                   
<NA>                         <NA>                         59
enterpreneur                 Unknown                      30
member                       Batanesi                     12
                             Mistretta                     6
executive                    Mistretta                     3
                             Batanesi                      3
member                       Mazzaroti                     2
boss                         Tortorici                     2
lawyer                       Unknown                       2
road haulier                 Unknown                       2
cooperating witness          Unknown                       1
boss                         Barcellona Pozzo di Gotto     1
                             Caltagirone                   1
boss Cosa Nostra in Messina  Unknown                       1
boss                         Mazzaroti                     1
                             B

In [ ]:
request_text = roles['Request'].astype('string').str.strip().str.lower()
arrest_values = {'house arrest', 'arrested', 'in jail'}
roles['arrested'] = request_text.isin(arrest_values).astype(int)

roles[['Node', 'Request', 'arrested']].head(20)

In [ ]:
assert roles.loc[roles['Request'].eq('arrested'), 'arrested'].eq(1).all()
assert roles.loc[roles['Request'].eq(' arrested'), 'arrested'].eq(1).all()
assert roles.loc[roles['Request'].eq('in jail'), 'arrested'].eq(1).all()
assert roles.loc[roles['Request'].eq('house arrest'), 'arrested'].eq(1).all()
assert roles.loc[roles['Request'].eq('arrest request denied'), 'arrested'].eq(0).all()

roles['arrested'].value_counts(dropna=False)